# FracAtlas pretraining — YOLO11n (GPU)

Trains the **FracAtlas binary fracture detector** used as a pretraining step for FemurFix Studio.

- **Task:** object detection, 1 class (`fracture`)
- **Data:** 2,868 images — 717 fractured (922 boxes) + 2,151 negatives
- **Split:** official FracAtlas split for positives (574/82/61) + 3x negatives per split
- **Model:** `yolo11n.pt`, 640 px, 50 epochs, horizontal flip disabled
- **Output:** `best.pt`, `best.onnx`, test metrics, confusion matrix, training curves

> **Scope:** FracAtlas mixes body parts (hands, wrists, elbows, shoulders, femurs) and has
> no region classes. This is a **pretraining/plumbing run**. The femoral-neck-vs-shaft
> classifier needs a hip/pelvis AP dataset annotated with
> `femoral_neck`, `intertrochanteric`, `subtrochanteric`, `femoral_shaft`, `distal_femur`.

Expected runtime: roughly 45–90 minutes for 50 epochs on a T4/P100. If you hit CUDA OOM,
lower `--batch` to 8.


In [ ]:
import subprocess
import sys

import torch

print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
else:
    print("WARNING: no GPU detected. Enable a GPU accelerator (T4/P100) in the notebook settings.")


In [ ]:
%pip install -q "ultralytics>=8.3,<9" "albumentations>=1.4.21,<2" "onnx>=1.16" "onnxslim>=0.1.31"
# Kaggle images ship albumentations 1.3.1, which lacks the quality_range argument Ultralytics uses.


## 1. Locate the repository

On Kaggle, attach the `femurfix-code` dataset. This cell walks up from the current directory and
also checks `/kaggle/input/femurfix-code`. If the repo is read-only (all Kaggle inputs are), it is
copied to `/kaggle/working/FemurFix` so training can write runs and checkpoints.


In [ ]:
import os
import shutil
import zipfile
from pathlib import Path


def find_code_root():
    candidates = [
        Path("/kaggle/input/femurfix-code"),
        Path("/kaggle/input/datasets/birwatkar/femurfix-code"),
    ]
    for candidate in candidates:
        if (candidate / "ml" / "train" / "train_yolo.py").is_file():
            return candidate
    for marker in Path("/kaggle/input").rglob("train_yolo.py"):
        if marker.parent.name == "train" and marker.parent.parent.name == "ml":
            return marker.parents[2]
    return None


CODE_INPUT = find_code_root()
if CODE_INPUT is not None and (CODE_INPUT / "code.zip").is_file():
    extracted_repo = Path("/kaggle/working/FemurFix")
    if not (extracted_repo / "ml" / "train" / "train_yolo.py").is_file():
        print(f"Extracting {CODE_INPUT / 'code.zip'} ...")
        with zipfile.ZipFile(CODE_INPUT / "code.zip") as archive:
            archive.extractall(extracted_repo)


def find_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents, Path("/kaggle/working/FemurFix")]
    if CODE_INPUT is not None:
        candidates.append(CODE_INPUT)
    for candidate in candidates:
        if (candidate / "ml" / "train" / "train_yolo.py").is_file():
            return candidate
    raise FileNotFoundError(
        "FemurFix repo not found. Clone/upload the repository or attach the femurfix-code dataset."
    )


REPO_ROOT = find_repo_root(Path.cwd())
if not os.access(REPO_ROOT, os.W_OK):
    writable_repo = Path("/kaggle/working/FemurFix")
    print(f"Copying read-only repo to {writable_repo} ...")
    shutil.copytree(REPO_ROOT, writable_repo, dirs_exist_ok=True)
    REPO_ROOT = writable_repo

ML_ROOT = REPO_ROOT / "ml"
RUN_ROOT = ML_ROOT / "runs"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
print("Repo:", REPO_ROOT)


## 2. Attach the dataset

**Prepared YOLO data (recommended):** upload the `ml/data` folder (only `images/` and `labels/`)
as a Kaggle Dataset, then point `PREPARED_DATA_DIR` at the attached path. It must contain
`images/{train,val,test}` and `labels/{train,val,test}`.

**Raw FracAtlas (alternative):** attach the raw FracAtlas dataset and set `RAW_FRACATLAS_DIR`.
The cell below will run the conversion into `/kaggle/working/fracatlas-yolo` using the official splits.

On Colab, replace the Kaggle paths with your mounted Drive paths.


In [ ]:
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

# Prepared YOLO dataset attached to this notebook. Kaggle mounts it either directly
# under /kaggle/input/<slug> or under /kaggle/input/datasets/<owner>/<slug>.
DATA_CANDIDATES = [
    Path("/kaggle/input/femurfix-fracatlas-yolo"),
    Path("/kaggle/input/datasets/birwatkar/femurfix-fracatlas-yolo"),
]
PREPARED_DATA_DIR = next(
    (
        candidate
        for candidate in DATA_CANDIDATES
        if (candidate / "images" / "train").is_dir() or (candidate / "dataset.zip").is_file()
    ),
    DATA_CANDIDATES[0],
)
if not (PREPARED_DATA_DIR / "images" / "train").is_dir() and not (PREPARED_DATA_DIR / "dataset.zip").is_file():
    for images_dir in Path("/kaggle/input").rglob("images"):
        if images_dir.is_dir() and (images_dir / "train").is_dir() and (images_dir.parent / "labels" / "train").is_dir():
            PREPARED_DATA_DIR = images_dir.parent
            break

# Alternative: raw FracAtlas attachment. Set this and leave PREPARED_DATA_DIR unused.
RAW_FRACATLAS_DIR = None  # e.g. Path("/kaggle/input/fracatlas/FracAtlas")

# Scratch data must not land in /kaggle/working, or it would be downloaded with the outputs.
WORK_ROOT = Path("/kaggle/temp") if Path("/kaggle/temp").is_dir() else Path("/kaggle/working")
SCRATCH_DATA = WORK_ROOT / "fracatlas-yolo"

if RAW_FRACATLAS_DIR is not None and Path(RAW_FRACATLAS_DIR).is_dir():
    DATA_ROOT = SCRATCH_DATA
    subprocess.run(
        [
            sys.executable,
            str(ML_ROOT / "scripts" / "convert_fracatlas.py"),
            "--src", str(RAW_FRACATLAS_DIR),
            "--dst", str(DATA_ROOT),
        ],
        check=True,
    )
elif (PREPARED_DATA_DIR / "images" / "train").is_dir():
    DATA_ROOT = PREPARED_DATA_DIR
elif (PREPARED_DATA_DIR / "dataset.zip").is_file():
    DATA_ROOT = SCRATCH_DATA
    if not (DATA_ROOT / "images" / "train").is_dir():
        print(f"Extracting {PREPARED_DATA_DIR / 'dataset.zip'} ...")
        with zipfile.ZipFile(PREPARED_DATA_DIR / "dataset.zip") as archive:
            archive.extractall(DATA_ROOT)
elif (ML_ROOT / "data" / "images" / "train").is_dir():
    DATA_ROOT = ML_ROOT / "data"
else:
    raise FileNotFoundError(
        "No YOLO dataset found. Attach the femurfix-fracatlas-yolo dataset, "
        "upload ml/data, or attach raw FracAtlas and set RAW_FRACATLAS_DIR."
    )

# Kaggle input is read-only and Ultralytics writes label caches next to the labels.
if str(DATA_ROOT).startswith("/kaggle/input") or not os.access(DATA_ROOT, os.W_OK):
    if not (SCRATCH_DATA / "images" / "train").is_dir():
        print(f"Copying read-only dataset to {SCRATCH_DATA} ...")
        shutil.copytree(DATA_ROOT, SCRATCH_DATA)
    DATA_ROOT = SCRATCH_DATA

print("Dataset root:", DATA_ROOT)
for split in ("train", "val", "test"):
    images = len(list((DATA_ROOT / "images" / split).glob("*")))
    labels = len(list((DATA_ROOT / "labels" / split).glob("*.txt")))
    print(f"  {split}: {images} images / {labels} labels")


In [ ]:
DATA_YAML = RUN_ROOT / "fracatlas_data.yaml"
DATA_YAML.write_text(
    f"""path: {Path(DATA_ROOT).resolve().as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: fracture
"""
)
print(DATA_YAML.read_text())


## 3. Train

The trainer runs validation every epoch, evaluates the held-out **test** split on the best
checkpoint, then exports `best.onnx` at 640 px. Remove `--cache` if RAM is tight.


In [ ]:
import subprocess
import sys

cmd = [
    sys.executable,
    str(ML_ROOT / "train" / "train_yolo.py"),
    "--data", str(DATA_YAML),
    "--device", "0",
    "--epochs", "50",
    "--imgsz", "640",
    "--batch", "16",
    "--workers", "4",
    "--cache",
    "--project", str(RUN_ROOT),
    "--name", "fracatlas_yolo11n",
    "--exist-ok",
]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


## 4. Inspect results


In [ ]:
import json

import pandas as pd
from IPython.display import Image, display

run_dir = RUN_ROOT / "fracatlas_yolo11n"
results_csv = run_dir / "results.csv"
if results_csv.is_file():
    df = pd.read_csv(results_csv)
    print(df.tail(3).to_string(index=False))
else:
    print("No results.csv found")

for name in ("results.png", "confusion_matrix.png", "val_batch0_pred.jpg"):
    path = run_dir / name
    if path.is_file():
        display(Image(filename=str(path)))

summary = run_dir / "run_summary.json"
if summary.is_file():
    print(json.dumps(json.loads(summary.read_text()), indent=2))


In [ ]:
from pathlib import Path
import shutil

run_dir = RUN_ROOT / "fracatlas_yolo11n"
weights = run_dir / "weights"
print("best.pt:", (weights / "best.pt").exists())
print("best.onnx:", (weights / "best.onnx").exists())

onnx_path = weights / "best.onnx"
if onnx_path.is_file():
    working = Path("/kaggle/working")
    target = (working if working.is_dir() else Path.cwd()) / "femur_fracture.onnx"
    shutil.copy2(onnx_path, target)
    print("Download-ready ONNX:", target)


## 5. Next steps

1. Copy `femur_fracture.onnx` into `backend/models/`.
2. Add local ONNX inference (`backend/app/services/ml_detector.py` + `onnxruntime`) and make
   `xray_analyzer.py` prefer it over the HF Space/heuristic.
3. Collect hip/pelvis AP films with region labels (`femoral_neck`, `intertrochanteric`,
   `subtrochanteric`, `femoral_shaft`, `distal_femur`) and fine-tune the detector. This is the
   model that fixes the neck-vs-shaft misclassification; the binary FracAtlas run does not.
